# CWRU plain patch Transformer
Shared window-split baseline. Run cells 1–9 in order. No normalization, resampling, new splits, CNN, or pretrained weights. Actual results are generated by your run.

64 tokens reduce each attention score matrix from 1024² to 64² entries (256× fewer); this is not a claim of 256× overall speedup. Learned positional embeddings retain patch order; mean pooling produces four class logits.


## 1. Imports, environment checks, seeds, and configuration

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import sys, json, platform, zipfile, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datetime import datetime, timezone

DATA_DIR = None  # Set to the complete six-file directory if discovery finds multiple matches.
OUTPUT_DIR = '/kaggle/working/transformer_results'
CONFIG = {
    'seed': 42, 'noise_seed': 2026, 'epochs': 30, 'batch_size': 64,
    'learning_rate': 0.001, 'benchmark_warmup': 10, 'benchmark_repeats': 50,
    'model': {'window_size': 1024, 'patch_size': 16, 'd_model': 64,
              'nhead': 4, 'layers': 2, 'dim_feedforward': 128, 'dropout': 0.1},
    'normalization': 'none', 'resampling': 'none', 'optimizer': 'Adam',
    'loss': 'CrossEntropyLoss', 'selection': 'highest validation accuracy; earliest tie',
    'precision': 'float32; AMP off; TF32 off', 'run_kind': 'full shared baseline',
}
# The local CLI can provide overrides; no supporting modules are needed on Kaggle.
overrides = globals().get('RUN_OVERRIDES', {})
DATA_DIR = overrides.get('data_dir', DATA_DIR)
OUTPUT_DIR = overrides.get('output_dir', OUTPUT_DIR)
CONFIG.update(overrides.get('config', {}))
"""Reproducible training with validation-only checkpoint selection."""
import os
import random
import time
import numpy as np
import pandas as pd
import torch
from torch import nn


def seed_everything(seed):
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)


def synchronize(device):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def train_model(model, loaders, config, device, output):
    if config['epochs'] < 1:
        raise ValueError('epochs must be at least 1')
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    history, best_accuracy, best_epoch = [], -1.0, None
    synchronize(device)
    start = time.perf_counter()
    for epoch in range(1, config['epochs'] + 1):
        record = {'epoch': epoch}
        for split in ('train', 'val'):
            model.train(split == 'train')
            loss_sum, correct, count = 0.0, 0, 0
            with torch.set_grad_enabled(split == 'train'):
                for x, y in loaders[split]:
                    x, y = x.to(device), y.to(device)
                    if split == 'train':
                        optimizer.zero_grad(set_to_none=True)
                    logits = model(x)
                    loss = criterion(logits, y)
                    if not torch.isfinite(loss):
                        raise FloatingPointError('Non-finite loss; inspect input scale and configuration.')
                    if split == 'train':
                        loss.backward()
                        optimizer.step()
                    loss_sum += loss.item() * len(y)
                    correct += (logits.argmax(1) == y).sum().item()
                    count += len(y)
            record[f'{split}_loss'] = loss_sum / count
            record[f'{split}_accuracy'] = correct / count
        if record['val_accuracy'] > best_accuracy:  # earliest epoch wins ties
            best_accuracy, best_epoch = record['val_accuracy'], epoch
            torch.save({'model_state_dict': model.state_dict(), 'model_config': config['model'],
                        'epoch': epoch, 'validation_accuracy': best_accuracy},
                       output / 'best_transformer.pth')
        history.append(record)
        print(record)
    synchronize(device)
    seconds = time.perf_counter() - start
    pd.DataFrame(history).to_csv(output / 'history.csv', index=False)
    return history, {'training_seconds': seconds, 'best_epoch': best_epoch,
                     'best_validation_accuracy': best_accuracy,
                     'training_timing_scope': 'Epoch loop including data loading, transfers, forward/backward, '
                     'optimizer updates, validation, checkpoint writes and logging; excludes setup, '
                     'history CSV export, test evaluation and benchmarking.'}

seed_everything(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output = Path(OUTPUT_DIR).resolve()
if output == Path('/kaggle/input') or Path('/kaggle/input') in output.parents:
    raise ValueError('OUTPUT_DIR must not be under read-only /kaggle/input')
output.mkdir(parents=True, exist_ok=True)
versions = {name: importlib.metadata.version(name) for name in
            ('numpy', 'torch', 'pandas', 'matplotlib')}
hardware = {'device': str(device), 'platform': platform.platform(),
            'processor': platform.processor(), 'cpu_threads': torch.get_num_threads(),
            'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
            'cuda_version': torch.version.cuda, 'cudnn_version': torch.backends.cudnn.version()}
print('Software:', versions, '\nHardware:', hardware)
print('Output:', output)


## 2. Discovering and validating the uploaded dataset

In [ ]:
"""Read shared splits without changing their membership or order."""
from pathlib import Path
import hashlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

CLASS_NAMES = ['Normal', 'Ball', 'Inner Race', 'Outer Race']
FILES = [f'{kind}_{split}.npy' for split in ('train', 'val', 'test') for kind in ('X', 'y')]


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def discover_data(data_dir=None, root='/kaggle/input'):
    if data_dir:
        folder = Path(data_dir)
        missing = [name for name in FILES if not (folder / name).is_file()]
        if missing:
            raise FileNotFoundError(f'{folder}: missing {missing}. Set DATA_DIR to the six-array folder.')
        return folder
    matches = sorted({p.parent for p in Path(root).rglob('X_train.npy')
                      if all((p.parent / name).is_file() for name in FILES)})
    if len(matches) != 1:
        raise ValueError(f'Expected one complete split folder under {root}; found {len(matches)}: '
                         f'{[str(p) for p in matches]}. Attach six files or set DATA_DIR explicitly.')
    return matches[0]


def load_splits(folder):
    arrays, manifest = {}, {}
    for split in ('train', 'val', 'test'):
        x_path, y_path = Path(folder) / f'X_{split}.npy', Path(folder) / f'y_{split}.npy'
        try:
            x, y = np.load(x_path, allow_pickle=False), np.load(y_path, allow_pickle=False)
        except Exception as exc:
            raise ValueError(f'Cannot load {split} arrays in {folder}: {exc}') from exc
        if x.ndim != 2 or x.shape[1] != 1024 or y.ndim != 1 or len(x) != len(y) or len(y) == 0:
            raise ValueError(f'{split}: expected nonempty X[N,1024], y[N]; got {x.shape}, {y.shape}')
        if x.dtype.kind not in 'fiu' or not np.isfinite(x).all():
            raise ValueError(f'{split}: X must contain finite real numbers')
        if y.dtype.kind not in 'iu' or not np.isin(y, range(4)).all():
            raise ValueError(f'{split}: y must have integer dtype and labels 0..3')
        with np.errstate(over='ignore'):
            converted = x.astype(np.float32)
        if not np.isfinite(converted).all():
            raise ValueError(f'{split}: values overflow float32')
        counts = np.bincount(y.astype(np.int64), minlength=4).tolist()
        print(f'{split}: X={x.shape}, y={y.shape}, counts={dict(zip(CLASS_NAMES, counts))}')
        arrays[split] = (converted, y.astype(np.int64))
        for path, a in ((x_path, x), (y_path, y)):
            manifest[path.name] = dict(sha256=sha256(path), shape=list(a.shape), dtype=str(a.dtype))
        manifest[y_path.name]['class_counts'] = counts
    return arrays, manifest


def make_loaders(arrays, batch_size, seed, device):
    generator = torch.Generator().manual_seed(seed)
    return {split: DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                             batch_size=batch_size, shuffle=(split == 'train'),
                             generator=generator if split == 'train' else None,
                             num_workers=0, pin_memory=(device.type == 'cuda'))
            for split, (x, y) in arrays.items()}

data_dir = discover_data(DATA_DIR)
arrays, manifest = load_splits(data_dir)
experiment = {'config': CONFIG, 'data_dir': str(data_dir), 'input_files': manifest,
              'python': sys.version, 'software_versions': versions, 'hardware': hardware,
              'utc_started': datetime.now(timezone.utc).isoformat(),
              'deterministic_algorithms': True,
              'data_protocol': 'Shared stratified seed-42 window split: 70/20/10; no new partition.',
              'limitations': 'Recordings may overlap splits; no load metadata; not cross-load evaluation.',
              'noise_protocol': 'PCG64 seed=2026+SNR by default (configurable base); float64 standard_normal '
              'draws for full test shape in row order, per-window mean(x**2), additive noise; float32 output; '
              'zero-power inputs remain zero. No renormalization of realized noise.'}
(output / 'experiment_config.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')
print(pd.DataFrame(manifest).T)


## 3. Dataset and DataLoader creation

In [ ]:
loaders = make_loaders(arrays, CONFIG['batch_size'], CONFIG['seed'], device)
print('Only training minibatch order is shuffled; split membership is never changed.')


## 4. Transformer definition and a forward-pass shape check

In [ ]:
"""Plain patch Transformer; no convolution or pretrained weights."""
import torch
from torch import nn


class PatchTransformer(nn.Module):
    def __init__(self, window_size=1024, patch_size=16, d_model=64,
                 nhead=4, layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        if window_size % patch_size or d_model % nhead:
            raise ValueError('Window must divide into patches; embedding must divide by heads.')
        self.window_size, self.patch_size = window_size, patch_size
        self.projection = nn.Linear(patch_size, d_model)
        self.position = nn.Parameter(torch.empty(1, window_size // patch_size, d_model))
        nn.init.normal_(self.position, std=0.02)
        # Independently initialized encoder layers.
        self.encoders = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout,
                                       batch_first=True) for _ in range(layers)])
        self.head = nn.Linear(d_model, 4)

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.window_size:
            raise ValueError(f'Expected [batch,{self.window_size}], got {tuple(x.shape)}')
        x = x.reshape(x.shape[0], -1, self.patch_size)  # [B,64,16]
        x = self.projection(x) + self.position         # [B,64,64]
        for encoder in self.encoders:
            x = encoder(x)
        return self.head(x.mean(dim=1))                # [B,4] logits

model = PatchTransformer(**CONFIG['model']).to(device)
model.eval()
with torch.inference_mode():
    check = model(torch.zeros(2, 1024, device=device))
assert check.shape == (2, 4)
print('Shape: [B,1024] -> [B,64,16] -> [B,64,64] -> mean pool [B,64] -> logits [B,4]')
print('Parameters:', sum(p.numel() for p in model.parameters()))


## 5. Training and checkpoint saving

In [ ]:
history, training = train_model(model, loaders, CONFIG, device, output)
print(training)


## 6. Reloading the best checkpoint and evaluating the test set

In [ ]:
"""Four-class metrics using NumPy only, including absent-class zero handling."""
import numpy as np


def confusion_matrix(y, pred, labels):
    result = np.zeros((len(labels), len(labels)), dtype=np.int64)
    np.add.at(result, (np.asarray(y, dtype=int), np.asarray(pred, dtype=int)), 1)
    return result


def accuracy_score(y, pred):
    return float(np.mean(np.asarray(y) == np.asarray(pred)))


def precision_recall_fscore_support(y, pred, labels, average=None, zero_division=0):
    cm = confusion_matrix(y, pred, labels)
    tp, support, predicted = np.diag(cm), cm.sum(1), cm.sum(0)
    precision = np.divide(tp, predicted, out=np.zeros(len(labels)), where=predicted != 0)
    recall = np.divide(tp, support, out=np.zeros(len(labels)), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros(len(labels)), where=(precision + recall) != 0)
    if average == 'macro':
        return precision.mean(), recall.mean(), f1.mean(), None
    if average is not None or zero_division != 0:
        raise ValueError('This baseline supports per-class/macro metrics and zero_division=0 only')
    return precision, recall, f1, support


def classification_report(y, pred, labels, target_names, output_dict=False, zero_division=0, digits=6):
    p, r, f, support = precision_recall_fscore_support(y, pred, labels, zero_division=zero_division)
    result = {name: {'precision': float(p[i]), 'recall': float(r[i]), 'f1-score': float(f[i]),
                     'support': int(support[i])} for i, name in enumerate(target_names)}
    result['macro avg'] = {'precision': float(p.mean()), 'recall': float(r.mean()),
                           'f1-score': float(f.mean()), 'support': int(support.sum())}
    result['weighted avg'] = {name: float(np.average(values, weights=support))
                              for name, values in [('precision', p), ('recall', r), ('f1-score', f)]}
    result['weighted avg']['support'] = int(support.sum())
    if output_dict:
        return result
    lines = [f'{"Class":16} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>8}']
    for name, row in result.items():
        lines.append(f'{name:16} {row["precision"]:10.{digits}f} {row["recall"]:10.{digits}f} '
                     f'{row["f1-score"]:10.{digits}f} {row["support"]:8d}')
    return '\n'.join(lines)
"""Evaluation, timing, plotting and deterministic per-window AWGN."""
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


@torch.inference_mode()
def predict(model, loader, device):
    model.eval()
    probabilities, targets = [], []
    for x, y in loader:
        probabilities.append(model(x.to(device)).softmax(1).cpu().numpy())
        targets.append(y.numpy())
    return np.concatenate(targets), np.concatenate(probabilities)


@torch.inference_mode()
def benchmark(model, x, device, batch_size=64, warmup=10, repeats=50):
    if warmup < 1 or repeats < 2:
        raise ValueError('Use at least one warmup and two timed repetitions')
    model.eval()
    result = {'device': str(device), 'precision': 'float32; AMP off; TF32 off',
              'warmup_iterations': warmup, 'repetitions': repeats,
              'timing_scope': 'Model forward only on resident device tensor; excludes loading, '
              'host/device transfer, softmax and metrics. Each call synchronized.',
              'input_selection': 'First min(requested batch size, test count) clean test windows'}
    for name, size in [('single', 1), ('batch', min(batch_size, len(x)))]:
        tensor = torch.from_numpy(x[:size].copy()).to(device)
        for _ in range(warmup):
            model(tensor)
        synchronize(device)
        durations = []
        for _ in range(repeats):
            synchronize(device)
            start = time.perf_counter()
            model(tensor)
            synchronize(device)
            durations.append(time.perf_counter() - start)
        mean = float(np.mean(durations))
        result[name] = {'batch_size': size, 'mean_batch_ms': mean * 1000,
                        'median_batch_ms': float(np.median(durations) * 1000),
                        'p95_batch_ms': float(np.percentile(durations, 95) * 1000),
                        'std_batch_ms': float(np.std(durations) * 1000),
                        'samples_per_second': size / mean,
                        'amortized_ms_per_sample': mean * 1000 / size}
    return result


def save_evaluation(y, probabilities, model, history, training, timing, output):
    pred = probabilities.argmax(1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, pred, labels=list(range(4)), average='macro', zero_division=0)
    metrics = {'accuracy': float(accuracy_score(y, pred)), 'macro_precision': float(precision),
               'macro_recall': float(recall), 'macro_f1': float(f1),
               'total_parameters': sum(p.numel() for p in model.parameters()),
               'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
               'test_samples': len(y), **training, 'inference': timing,
               'protocol': 'shared window-split baseline; no cross-load evaluation',
               'metric_convention': 'Fractions 0..1; all four labels; zero_division=0'}
    (output / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    report = classification_report(y, pred, labels=list(range(4)), target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(output / 'classification_report.csv', index_label='class')
    (output / 'metrics.txt').write_text(json.dumps(metrics, indent=2) + '\n\n' +
        classification_report(y, pred, labels=list(range(4)), target_names=CLASS_NAMES,
                              digits=6, zero_division=0), encoding='utf-8')
    predictions = pd.DataFrame({'test_row_index': np.arange(len(y)), 'true_class': y, 'predicted_class': pred})
    for i, name in enumerate(CLASS_NAMES):
        predictions[f'probability_{i}_{name.replace(" ", "_")}'] = probabilities[:, i]
    predictions.to_csv(output / 'test_predictions.csv', index=False)
    counts = confusion_matrix(y, pred, labels=list(range(4)))
    normalized = np.divide(counts, counts.sum(1, keepdims=True),
                           out=np.zeros_like(counts, dtype=float), where=counts.sum(1, keepdims=True) != 0)
    for name, matrix in [('counts', counts), ('normalized', normalized)]:
        pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
            output / f'confusion_matrix_{name}.csv', index_label='true_class')
        fig, ax = plt.subplots(figsize=(7, 6))
        img = ax.imshow(matrix, cmap='Blues', vmin=0, vmax=1 if name == 'normalized' else None)
        fig.colorbar(img, ax=ax)
        ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES,
               yticklabels=CLASS_NAMES, xlabel='Predicted class', ylabel='True class',
               title=f'Transformer test confusion matrix ({name})')
        for row in range(4):
            for col in range(4):
                ax.text(col, row, f'{matrix[row,col]:.3f}' if name == 'normalized' else str(matrix[row,col]),
                        ha='center', va='center', color='white' if matrix[row,col] > matrix.max()/2 else 'black')
        fig.tight_layout()
        fig.savefig(output / f'confusion_matrix_{name}.png', dpi=180)
        plt.show()
        plt.close(fig)
    frame = pd.DataFrame(history)
    for metric in ('loss', 'accuracy'):
        fig, ax = plt.subplots(figsize=(7, 4))
        for split in ('train', 'val'):
            ax.plot(frame.epoch, frame[f'{split}_{metric}'], label=split)
        ax.set(xlabel='Epoch', ylabel=metric.title(), title=f'Transformer {metric}')
        ax.legend()
        fig.tight_layout()
        fig.savefig(output / f'{metric}_curves.png', dpi=180)
        plt.show()
        plt.close(fig)
    row = {k: metrics[k] for k in ('accuracy', 'macro_precision', 'macro_recall', 'macro_f1',
                                  'total_parameters', 'trainable_parameters', 'training_seconds')}
    row.update(model='Transformer', device=timing['device'], precision=timing['precision'],
               single_latency_ms=timing['single']['mean_batch_ms'],
               throughput_samples_per_second=timing['batch']['samples_per_second'],
               throughput_batch_size=timing['batch']['batch_size'],
               amortized_ms_per_sample=timing['batch']['amortized_ms_per_sample'],
               timing_scope=timing['timing_scope'], protocol=metrics['protocol'])
    pd.DataFrame([row]).to_csv(output / 'comparison_summary.csv', index=False)
    return metrics


def add_awgn(x, snr_db, seed):
    # One PCG64 generator per SNR, float64 draws in C row order, then float32 output.
    # Do not normalize the realized noise: formula specifies expected noise power.
    x64 = np.asarray(x, dtype=np.float64)
    power = np.mean(x64 ** 2, axis=1, keepdims=True)
    rng = np.random.Generator(np.random.PCG64(seed))
    noise = rng.standard_normal(x64.shape) * np.sqrt(power / (10.0 ** (snr_db / 10.0)))
    return (x64 + noise).astype(np.float32)  # zero-power rows stay exactly zero


def evaluate_noise(model, arrays, clean_metrics, device, config, output):
    from torch.utils.data import TensorDataset, DataLoader
    x, y = arrays['test']
    rows = [dict(condition='clean', snr_db=None, noise_seed=None,
                 accuracy=clean_metrics['accuracy'], macro_f1=clean_metrics['macro_f1'])]
    for snr in (20, 10, 5, 0):
        seed = config['noise_seed'] + snr
        noisy = add_awgn(x, snr, seed)
        loader = DataLoader(TensorDataset(torch.from_numpy(noisy), torch.from_numpy(y)),
                            batch_size=config['batch_size'], shuffle=False)
        targets, probs = predict(model, loader, device)
        pred = probs.argmax(1)
        f1 = precision_recall_fscore_support(targets, pred, labels=list(range(4)),
                                            average='macro', zero_division=0)[2]
        rows.append(dict(condition=f'{snr} dB', snr_db=snr, noise_seed=seed,
                         accuracy=float(accuracy_score(targets, pred)), macro_f1=float(f1)))
    frame = pd.DataFrame(rows)
    frame.to_csv(output / 'noise_robustness.csv', index=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    for metric in ('accuracy', 'macro_f1'):
        ax.plot(frame.condition, frame[metric], marker='o', label=metric)
    ax.set(xlabel='Test condition', ylabel='Score', ylim=(0, 1.02), title='Clean-trained Transformer robustness')
    ax.legend()
    fig.tight_layout()
    fig.savefig(output / 'noise_robustness.png', dpi=180)
    plt.show()
    plt.close(fig)
    return frame

checkpoint = torch.load(output / 'best_transformer.pth', map_location=device, weights_only=True)
model = PatchTransformer(**checkpoint['model_config']).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
targets, probabilities = predict(model, loaders['test'], device)
assert np.array_equal(targets, arrays['test'][1])
assert np.isfinite(probabilities).all()
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
timing = benchmark(model, arrays['test'][0], device, CONFIG['batch_size'],
                   CONFIG['benchmark_warmup'], CONFIG['benchmark_repeats'])


## 7. Metrics, tables, and plots

In [ ]:
metrics = save_evaluation(targets, probabilities, model, history, training, timing, output)
experiment['training'] = training
experiment['inference'] = timing
(output / 'experiment_config.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')
print((output / 'metrics.txt').read_text(encoding='utf-8'))
print(pd.read_csv(output / 'comparison_summary.csv').to_string(index=False))


## 8. Noise robustness evaluation

In [ ]:
noise_results = evaluate_noise(model, arrays, metrics, device, CONFIG, output)
print(noise_results.to_string(index=False))
print('Test robustness is descriptive evaluation only; do not tune or select checkpoints from it.')


## 9. Packaging results for download

In [ ]:
artifact_hashes = {p.name: sha256(p) for p in sorted(output.iterdir())
                   if p.is_file() and p.name != 'artifact_hashes.json'}
(output / 'artifact_hashes.json').write_text(json.dumps(artifact_hashes, indent=2), encoding='utf-8')
archive = output.parent / (output.name + '.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        if path.is_file():
            bundle.write(path, arcname=path.name)
print('Results:', output, '\nDownload ZIP:', archive)
try:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))
except ImportError:
    pass
